In [69]:
import os
import chromadb
from pypdf import PdfReader
from openai import OpenAI
from dotenv import load_dotenv
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [70]:
import langchain
print(langchain.__version__)

import sys
print(sys.executable)

1.3.13
c:\Users\thubz\Vector datase\vector-db-projects\.venv\Scripts\python.exe


In [71]:
load_dotenv()
Api_key = os.getenv("OPENAI_API_KEY")
openai_client =OpenAI(api_key=Api_key)



In [72]:
client = chromadb.PersistentClient(path="./chromdb")

In [73]:
reader = PdfReader("Employee-handbook.pdf")
text = ""

for page in reader.pages:
    text += page.extract_text()
print(text)

ABC Corporation Employee Handbook 
Welcome 
Welcome to ABC Corporation. We are committed to creating a professional, inclusive, and 
productive workplace. This handbook outlines our policies, procedures, and employee 
benefits. All employees are expected to read and follow these guidelines. 
 
Company Values 
Our core values include: 
• Integrity 
• Respect 
• Innovation 
• Teamwork 
• Customer Focus 
• Accountability 
Employees are expected to demonstrate these values in their daily work. 
 
Employment Types 
ABC Corporation employs staff under the following categories: 
• Full-time Employees 
• Part-time Employees 
• Contract Employees 
• Temporary Employees 
• Interns 
Benefits vary depending on employment type. 
 
Working Hours 
The standard workweek is Monday through Friday. Normal working hours are 9:00 AM to 5:00 PM. 
Employees receive one hour for lunch. 
Employees are expected to arrive on time. 
Flexible work arrangements may be approved by managers depending on business 
req

In [74]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100
)

chunks = splitter.split_text(text)
print(f"Total chunks: {len(chunks)}")

for i, chunk in enumerate(chunks, start=1):
    print(f"\n===== Chunk {i} =====")
    print(chunk)

Total chunks: 19

===== Chunk 1 =====
ABC Corporation Employee Handbook 
Welcome 
Welcome to ABC Corporation. We are committed to creating a professional, inclusive, and 
productive workplace. This handbook outlines our policies, procedures, and employee 
benefits. All employees are expected to read and follow these guidelines. 
 
Company Values 
Our core values include: 
• Integrity 
• Respect 
• Innovation 
• Teamwork 
• Customer Focus 
• Accountability 
Employees are expected to demonstrate these values in their daily work.

===== Chunk 2 =====
• Accountability 
Employees are expected to demonstrate these values in their daily work. 
 
Employment Types 
ABC Corporation employs staff under the following categories: 
• Full-time Employees 
• Part-time Employees 
• Contract Employees 
• Temporary Employees 
• Interns 
Benefits vary depending on employment type. 
 
Working Hours 
The standard workweek is Monday through Friday. Normal working hours are 9:00 AM to 5:00 PM. 
Employees rece

In [90]:
# OpenAI embedding function
embedding_function = OpenAIEmbeddingFunction(
    api_key=Api_key ,
    model_name="text-embedding-3-small"
)

In [91]:
#remove old collection during testing
collection_name = "employee-handbook"
try:
    client.delete_collection("collection_name")
except:
    print("collection doesnt exist yet.")

In [92]:
#create collection
collection = client.get_or_create_collection(
    name="collection_name",
    embedding_function=embedding_function
)

In [93]:
#storing chunks in chromadb
collection.add(
    ids=[
        str(i) for i in range(len(chunks))
    ],
    documents=chunks,
    metadatas=[
        {
            "source": "Employee-handbook.pdf",
            "chunk": i
        }
        for i in range(len(chunks))
    ]
)

In [94]:
query = "Tell me all about perfomace"

In [95]:
results = collection.query(
        query_texts=[query],
        n_results=3
    )

retrieved_documents = results["documents"][0]


context = "\n\n".join(
        retrieved_documents
    )


In [96]:
prompt = f"""
You are an assistant that answers questions using only the provided document.

If the answer is not found in the context, say:
"I could not find this information in the document."

Context:

{context}


Question:

{query}


Answer:
"""


response = openai_client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

print(response.choices[0].message.content)

Performance reviews at ABC Corporation are conducted annually. During these reviews, employees receive feedback on several key areas including:

- Productivity
- Teamwork
- Communication
- Technical Skills
- Professional Development

Additionally, performance reviews may influence salary increases and promotions.
